# Importing libraries

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os
import cv2
import pandas as pd
import seaborn as sns


# Data preprocessing

## Training image preprocessing

In [ ]:
import tensorflow as tf

# Data augmentation layer
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),  # Randomly flip images
    tf.keras.layers.RandomRotation(0.2),                   # Rotate images up to 20%
    tf.keras.layers.RandomZoom(0.2, 0.2),                   # Random zoom in/out
    tf.keras.layers.RandomBrightness(0.2),                 # Adjust brightness randomly
    tf.keras.layers.RandomContrast(0.2),                   # Adjust contrast randomly
    tf.keras.layers.Rescaling(1./255)                      # Normalize pixel values to [0, 1]
])

# Load training dataset
training_set = tf.keras.utils.image_dataset_from_directory(
    'train',
    labels="inferred",
    label_mode="categorical",
    color_mode="rgb",
    batch_size=32,
    image_size=(128, 128),
    shuffle=True,
    seed=123,  # Seed for reproducibility
)
train_class_names = training_set.class_names



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get the labels from the dataset (each batch will contain labels for images)
labels = np.concatenate([y.numpy() for x, y in training_set], axis=0)

# Convert one-hot encoding to class indices
class_indices = np.argmax(labels, axis=1)

# Get the counts of each class
class_counts = np.bincount(class_indices)

# Plot the distribution with increased bar width
plt.figure(figsize=(15, 6))
bars = plt.bar(range(len(train_class_names)), class_counts, tick_label=train_class_names, width=0.9)  # Increase width here
plt.xlabel('Class')
plt.ylabel('Number of Samples')
plt.title('Class Distribution in Training Set')
plt.xticks(rotation=90)  # Rotate class names for better visibility

# Add numbers on top of the bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval + 2,  # Adjust position slightly above the bar
             str(int(yval)), ha='center', va='bottom', fontsize=7)

plt.show()


In [ ]:
training_set = training_set.map(lambda x, y: (data_augmentation(x, training=True), y))

# Display some augmented images
import matplotlib.pyplot as plt

for images, labels in training_set.take(1):  # Take one batch
    plt.figure(figsize=(12, 8))
    for i in range(9):  # Show 9 images
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy())
        plt.axis("off")
    plt.show()

## Validation image preprocessing

In [ ]:
validation_set = tf.keras.utils.image_dataset_from_directory(
    'valid',
    labels="inferred",
    label_mode="categorical",
    color_mode="rgb",
    batch_size=32,
    image_size=(128, 128),
    shuffle=True,  # No shuffling for validation to maintain consistency
    seed=123,       # Seed for reproducibility
)




In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get the labels from the validation dataset (each batch will contain labels for images)
labels = np.concatenate([y.numpy() for x, y in validation_set], axis=0)

# Convert one-hot encoding to class indices
class_indices = np.argmax(labels, axis=1)

# Get the counts of each class
class_counts = np.bincount(class_indices)

# Plot the distribution with increased bar width
plt.figure(figsize=(15, 6))
bars = plt.bar(range(len(train_class_names)), class_counts, tick_label=train_class_names, width=0.9)  # Increase width here
plt.xlabel('Class')
plt.ylabel('Number of Samples')
plt.title('Class Distribution in Validation Set')
plt.xticks(rotation=90)  # Rotate class names for better visibility

# Add numbers on top of the bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval + 2,  # Adjust position slightly above the bar
             str(int(yval)), ha='center', va='bottom', fontsize=7)

plt.show()


In [ ]:
# Normalize pixel values
normalization_layer = tf.keras.layers.Rescaling(1./255)

validation_set = validation_set.map(lambda x, y: (normalization_layer(x), y))

# Display some validation images
import matplotlib.pyplot as plt

for images, labels in validation_set.take(1):  # Take one batch
    plt.figure(figsize=(12, 8))
    for i in range(9):  # Show 9 images
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy())
        plt.axis("off")
    plt.show()

# Building Model

## CNN

In [ ]:
from tensorflow.keras.layers import Dense,Conv2D,MaxPooling2D,Flatten,Dropout
from tensorflow.keras.models import Sequential

In [ ]:
model = Sequential()

In [ ]:
model.add(Conv2D(filters=32,kernel_size=3,padding='same', activation='relu',input_shape=(128,128,3)))
model.add(Conv2D(filters=32,kernel_size=3, activation='relu'))
model.add(MaxPooling2D(pool_size=2,strides=2))

In [ ]:
model.add(Conv2D(filters=64,kernel_size=3,padding='same', activation='relu'))
model.add(Conv2D(filters=64,kernel_size=3, activation='relu'))
model.add(MaxPooling2D(pool_size=2,strides=2))

In [ ]:
model.add(Conv2D(filters=128,kernel_size=3,padding='same', activation='relu'))
model.add(Conv2D(filters=128,kernel_size=3, activation='relu'))
model.add(MaxPooling2D(pool_size=2,strides=2))

In [ ]:
model.add(Conv2D(filters=256,kernel_size=3,padding='same', activation='relu'))
model.add(Conv2D(filters=256,kernel_size=3, activation='relu'))
model.add(MaxPooling2D(pool_size=2,strides=2))

In [ ]:
model.add(Conv2D(filters=512,kernel_size=3,padding='same', activation='relu'))
model.add(Conv2D(filters=512,kernel_size=3, activation='relu'))
model.add(MaxPooling2D(pool_size=2,strides=2))

In [ ]:
model.add(Dropout(0.25))

In [ ]:
model.add(Flatten())

In [ ]:
model.add(Dense(units=1500,activation='relu'))

In [ ]:
model.add(Dropout(0.4))

In [ ]:
model.add(Dense(units=38,activation='softmax'))

### Compile the model

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),loss='categorical_crossentropy',metrics=['accuracy'])

In [ ]:
model.summary()

### Model Training

In [ ]:
training_history = model.fit(training_set,validation_data=validation_set,epochs=20)

### Model Evaluation

In [ ]:
train_loss, train_accuracy = model.evaluate(training_set)

In [ ]:
print(f"Train Loss: {train_loss}")
print(f"Train Accuracy: {train_accuracy}")

In [ ]:
validation_loss, validation_accuracy = model.evaluate(validation_set)

In [ ]:
print(f"Validation Loss: {validation_loss}")
print(f"Validation Accuracy: {validation_accuracy}")

In [ ]:
model.save('base_model_CNN.h5')

In [ ]:
import json
with open('training_history_base_model_CNN.json', 'w') as f:
    json.dump(str(training_history.history), f)

In [ ]:
# Accuracy plot
plt.plot(training_history.history['accuracy'], label = 'accuracy', color='blue')
plt.plot(training_history.history['val_accuracy'], label = 'val_accuracy', color='red')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.legend(loc='lower right')
plt.show()

## VGG16

### Compile and train the model

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models

# Load the VGG16 model without the top layers
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

# Freeze the convolutional base
base_model.trainable = False

# Create a new model on top of the base
model = models.Sequential([
    base_model,  # Add the pre-trained VGG16 base
    layers.Flatten(),  # Flatten the output
    layers.Dense(256, activation='relu'),  # Fully connected layer
    layers.Dropout(0.5),  # Dropout for regularization
    layers.Dense(38, activation='softmax')  # Output layer
])

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
training_history = model.fit(
    training_set,
    validation_data=validation_set,
    epochs=20
)


In [ ]:
# Save the model
model.save('vgg16_finetuned_model.h5')

### Model Evaluation

In [ ]:
train_loss, train_accuracy = model.evaluate(training_set)


In [ ]:
print(f"Train Loss: {train_loss}")
print(f"Train Accuracy: {train_accuracy}")

In [ ]:
validation_loss, validation_accuracy = model.evaluate(validation_set)

In [ ]:
print(f"Validation Loss: {validation_loss}")
print(f"Validation Accuracy: {validation_accuracy}")

In [ ]:
# Accuracy plot
plt.plot(training_history.history['accuracy'], label = 'accuracy', color='blue')
plt.plot(training_history.history['val_accuracy'], label = 'val_accuracy', color='red')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.legend(loc='lower right')
plt.show()